## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# a running log of every row dropped, printed at the end
drop_log = []

def log_drop(stage, before, after):
    dropped = before - after
    drop_log.append({
        "stage": stage,
        "dropped": dropped,
        "remaining": after,
        "pct_of_original": round(dropped / before * 100, 2)
    })
    print(f"{stage:45s} dropped {dropped:>7,}  remaining {after:>8,}")

In [2]:
ppr = pd.read_csv(RAW / "ppr_raw.csv", encoding="latin-1")
n_original = len(ppr)

print(f"Loaded {n_original:,} rows")
print(f"Columns: {list(ppr.columns)}")

Loaded 799,067 rows
Columns: ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (\x80)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']


C:\Users\heffo\AppData\Local\Temp\ipykernel_26876\1920956887.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ppr = pd.read_csv(RAW / "ppr_raw.csv", encoding="latin-1")


## 2. Rename and Parse

In [3]:
# Renaming columns

ppr = ppr.rename(columns={
    "Date of Sale (dd/mm/yyyy)":  "date",
    "Address":                    "address",
    "County":                     "county",
    "Eircode":                    "eircode",
    "Price (\x80)":               "price_raw",
    "Not Full Market Price":      "not_full_market",
    "VAT Exclusive":              "vat_exclusive",
    "Description of Property":    "description",
    "Property Size Description":  "size_description",
})

print(list(ppr.columns))

['date', 'address', 'county', 'eircode', 'price_raw', 'not_full_market', 'vat_exclusive', 'description', 'size_description']


In [4]:
# Parse price and date 

ppr["price"] = (ppr["price_raw"]
                .str.replace("\x80", "", regex=False)
                .str.replace(",", "", regex=False)
                .str.strip()
                .astype(float))

ppr["date"] = pd.to_datetime(ppr["date"], format="%d/%m/%Y")
ppr["year"] = ppr["date"].dt.year
ppr["month"] = ppr["date"].dt.to_period("M")

print(f"Date range: {ppr['date'].min().date()} to {ppr['date'].max().date()}")
print(f"\nPrice summary:")
print(ppr["price"].describe().apply(lambda v: f"{v:,.0f}").to_string())

Date range: 2010-01-01 to 2026-07-31

Price summary:
count        799,067
mean         319,174
std        1,234,377
min            5,001
25%          145,000
50%          243,500
75%          365,000
max      387,665,198


In [5]:
print("Ten highest:")
print(ppr.nlargest(10, "price")[["date", "address", "county", "price"]].to_string())

print("\nTen lowest:")
print(ppr.nsmallest(10, "price")[["date", "address", "county", "price"]].to_string())

Ten highest:
             date                                                                      address   county         price
690730 2024-10-17                                  24 Tinakilly Grove, Tinakilly Park, Rathnew  Wicklow  3.876652e+08
586434 2023-02-10                                     O'Devaney Gardens, Arbour Hill, Dublin 7   Dublin  2.250000e+08
771088 2026-01-27                                                         Montpelier, Dublin 7   Dublin  2.250000e+08
706188 2024-12-23                                       Cooper Square, Seven Mills, Clonburris   Dublin  2.219427e+08
780814 2026-03-31                                            BLOCK A  B AND C, NEWMARKET YARDS   Dublin  1.893925e+08
431243 2020-07-17  Apartments 1 - 186 Cheevers Court, Apartments 1-182 Haliday House, Cualanor   Dublin  1.823789e+08
473413 2021-04-15                                          8th Lock, Ratoath Road, Pelletstown   Dublin  1.701428e+08
749598 2025-09-30                     Block

In [6]:
print("Price distribution tails:")
for q in [0.0001, 0.001, 0.005, 0.01, 0.05, 0.5, 0.95, 0.99, 0.995, 0.999, 0.9999]:
    print(f"  {q:>8.2%}  {ppr['price'].quantile(q):>15,.0f}")

Price distribution tails:
     0.01%            5,714
     0.10%            8,000
     0.50%           16,667
     1.00%           24,000
     5.00%           50,100
    50.00%          243,500
    95.00%          700,000
    99.00%        1,412,500
    99.50%        2,000,000
    99.90%        6,239,804
    99.99%       50,690,876


In [7]:
extremes = ppr[(ppr["price"] < 20_000) | (ppr["price"] > 2_000_000)]
print(f"Rows outside 20k - 2m: {len(extremes):,}  ({len(extremes)/len(ppr)*100:.2f}%)")
print(f"\nOf which flagged not-full-market:")
print(extremes["not_full_market"].value_counts().to_string())

print(f"\nSplit by tail:")
low = ppr[ppr["price"] < 20_000]
high = ppr[ppr["price"] > 2_000_000]
print(f"  below 20k:  {len(low):,}  ({(low['not_full_market']=='Yes').sum():,} flagged)")
print(f"  above 2m:   {len(high):,}  ({(high['not_full_market']=='Yes').sum():,} flagged)")

Rows outside 20k - 2m: 8,836  (1.11%)

Of which flagged not-full-market:
not_full_market
No     7603
Yes    1233

Split by tail:
  below 20k:  4,888  (1,029 flagged)
  above 2m:   3,948  (204 flagged)


## 3. Covert Irish language to English

In [8]:
print("Before mapping:")
print(ppr["description"].value_counts(dropna=False).to_string())

Before mapping:
description
Second-Hand Dwelling house /Apartment    656283
New Dwelling house /Apartment            142735
Teach/Árasán Cónaithe Atháimhe               45
Teach/Árasán Cónaithe Nua                     3
Teach/?ras?n C?naithe Nua                     1


In [9]:
# anything containing "Nua" (new) or the mojibake equivalent is a new dwelling,
# anything containing "Atháimhe" / "Ath" is second-hand

def map_description(val):
    v = str(val)
    if "New Dwelling" in v or "Nua" in v:
        return "New"
    if "Second-Hand" in v or "Ath" in v or "th\u00e1imhe" in v:
        return "Second-Hand"
    return np.nan

ppr["property_type"] = ppr["description"].apply(map_description)

print("After mapping:")
print(ppr["property_type"].value_counts(dropna=False).to_string())

After mapping:
property_type
Second-Hand    656328
New            142739


## 4. VAT Adjustment

In [10]:
# new residential property prices in the PPR are filed excluding VAT
# Irish VAT on new residential property is 13.5%

VAT_RATE = 0.135

ppr["price_incl_vat"] = np.where(
    ppr["vat_exclusive"] == "Yes",
    ppr["price"] * (1 + VAT_RATE),
    ppr["price"]
)

n_adjusted = (ppr["vat_exclusive"] == "Yes").sum()
print(f"VAT adjustment applied to {n_adjusted:,} sales")

comparison = ppr.groupby("vat_exclusive").agg(
    n=("price", "size"),
    median_before=("price", "median"),
    median_after=("price_incl_vat", "median"),
)
print(f"\n{comparison.to_string()}")

VAT adjustment applied to 140,374 sales

                    n  median_before  median_after
vat_exclusive                                     
No             658693       226000.0    226000.000
Yes            140374       308369.0    349998.815


In [11]:
print(pd.crosstab(ppr["property_type"], ppr["vat_exclusive"]))

vat_exclusive      No     Yes
property_type                
New              2365  140374
Second-Hand    656328       0


In [12]:
new_only = ppr[ppr["property_type"] == "New"]
print(new_only.groupby("vat_exclusive")["price"].describe()[["count", "25%", "50%", "75%"]].round(0).to_string())

                  count       25%       50%       75%
vat_exclusive                                        
No               2365.0   80000.0  158546.0  350000.0
Yes            140374.0  220264.0  308369.0  396476.0


2,365 sales are described as new dwellings but not flagged VAT-exclusive (1.7% of new builds). Their price distribution sits well below VAT-exclusive new builds rather than above, so these are unlikely to be gross-priced new sales. They are more plausibly non-standard transactions such as local authority or affordable housing transfers. No VAT adjustment is applied, consistent with keying the adjustment off the VAT flag rather than the property description.

## 5. Filtering

In [13]:
n = len(ppr)

# 1. non-arms-length transactions
ppr = ppr[ppr["not_full_market"] == "No"].copy()
log_drop("Not full market price", n, len(ppr)); n = len(ppr)

# 2. restrict to individual dwellings
#    upper bound removes bulk and portfolio transactions
#    lower bound removes nominal transfers
PRICE_MIN, PRICE_MAX = 20_000, 2_000_000
ppr = ppr[ppr["price_incl_vat"].between(PRICE_MIN, PRICE_MAX)].copy()
log_drop(f"Price outside {PRICE_MIN:,} - {PRICE_MAX:,}", n, len(ppr)); n = len(ppr)

# 3. placeholder Eircodes - blank the code, keep the sale
PLACEHOLDERS = ["A123456", "A00AA00"]
mask = ppr["eircode"].isin(PLACEHOLDERS)
print(f"\nPlaceholder Eircodes blanked: {mask.sum():,}")
ppr.loc[mask, "eircode"] = np.nan

Not full market price                         dropped  40,721  remaining  758,346
Price outside 20,000 - 2,000,000              dropped   7,686  remaining  750,660

Placeholder Eircodes blanked: 154


In [14]:
survivors = ppr[(ppr["property_type"] == "New") & (ppr["vat_exclusive"] == "No")]
print(f"Remaining after filtering: {len(survivors):,}")

Remaining after filtering: 1,901


In [15]:
print(f"Rows now: {len(ppr):,}")
print(f"Drop log entries: {len(drop_log)}")

Rows now: 750,660
Drop log entries: 2


In [16]:
print(pd.DataFrame(drop_log).to_string(index=False))

                           stage  dropped  remaining  pct_of_original
           Not full market price    40721     758346             5.10
Price outside 20,000 - 2,000,000     7686     750660             1.01


## 6. Eircode and Routing Key

In [17]:
# Dublin 6W is the one exception to the standard letter-digit-digit routing key format

EIRCODE_PATTERN = r"^([A-Za-z]\d{2}|D6W)\s?[A-Za-z0-9]{4}$"

ppr["eircode"] = ppr["eircode"].str.upper().str.replace(" ", "", regex=False)
valid = ppr["eircode"].str.match(EIRCODE_PATTERN, na=False)

print(f"Eircode present:  {ppr['eircode'].notna().sum():,}")
print(f"Format-valid:     {valid.sum():,}")
print(f"Malformed:        {(ppr['eircode'].notna() & ~valid).sum():,}")

# blank anything malformed rather than dropping the sale
ppr.loc[ppr["eircode"].notna() & ~valid, "eircode"] = np.nan

Eircode present:  229,756
Format-valid:     229,756
Malformed:        0


In [18]:
ppr["routing_key"] = ppr["eircode"].str[:3]

print(f"Sales with a routing key: {ppr['routing_key'].notna().sum():,}")
print(f"Distinct routing keys:    {ppr['routing_key'].nunique()}")
print(f"\nCoverage by year:")

cov = ppr.groupby("year").agg(
    sales=("routing_key", "size"),
    with_rk=("routing_key", "count"),
)
cov["pct"] = (cov["with_rk"] / cov["sales"] * 100).round(1)
print(cov.to_string())

Sales with a routing key: 229,756
Distinct routing keys:    302

Coverage by year:
      sales  with_rk   pct
year                      
2010  19777        1   0.0
2011  17215        2   0.0
2012  23411        0   0.0
2013  27888       21   0.1
2014  42047       52   0.1
2015  46870       38   0.1
2016  47461       65   0.1
2017  51848       97   0.2
2018  53674      105   0.2
2019  54821      161   0.3
2020  46455      309   0.7
2021  56480    29146  51.6
2022  59263    45764  77.2
2023  59187    45409  76.7
2024  56932    43074  75.7
2025  58257    43359  74.4
2026  29074    22153  76.2


In [19]:
# placeholder removal should substantially reduce apparent cross-county spread

xref = (ppr.dropna(subset=["routing_key"])
        .groupby("routing_key")["county"]
        .nunique()
        .sort_values(ascending=False))

print(f"Routing keys spanning multiple counties: {(xref > 1).sum()} of {len(xref)}")
print(f"\nMost cross-county:")
print(xref.head(15).to_string())

Routing keys spanning multiple counties: 142 of 302

Most cross-county:
routing_key
W91    14
V94    14
N41    12
H91    12
F91    11
N91    11
A96    11
W23    10
C15    10
R32     9
R93     8
F12     8
A94     8
R95     8
H12     8


In [20]:
rk = "W91"
sample = ppr[ppr["routing_key"] == rk]

print(f"{rk}: {len(sample):,} sales, {sample['eircode'].nunique():,} distinct Eircodes")
print(f"\nCounty spread:")
print(sample["county"].value_counts().to_string())
print(f"\nAddresses from the smaller counties:")
minor = sample["county"].value_counts().tail(8).index
print(sample[sample["county"].isin(minor)][["address", "county", "eircode"]].head(15).to_string())

W91: 4,350 sales, 4,135 distinct Eircodes

County spread:
county
Kildare      3566
Wicklow       764
Kilkenny        4
Dublin          4
Carlow          2
Donegal         2
Wexford         1
Cavan           1
Laois           1
Louth           1
Offaly          1
Westmeath       1
Clare           1
Meath           1

Addresses from the smaller counties:
                                               address     county  eircode
546738            COIS NA MARA, GRANGE, FETHARD ON SEA    Wexford  W91E1XK
567146                         URBAL, KILNALECK, CAVAN      Cavan  W91E299
634061  5 The Crescent, Mount Stewart, Stradbally Road      Laois  W91X8K3
648212              2 NEWTOWN PLACE, THE MEADOWS, KILL      Louth  W91F9PN
662351                       GRANGE, EDENDERRY, OFFALY     Offaly  W91PR64
673116     58 BURGAGE CASTLE, BLESSINGTON, CO. WICKLOW  Westmeath  W91H58A
715167            7 THE MILLICENT, COIS ABHAINN, CLANE      Clare  W91W324
725018                      KILTALE, DUNSANY,

In [21]:
# Routing key to dominant county 

# the county field contains occasional data entry errors, visible
# where the address text and Eircode both contradict it. Assigning
# each routing key its dominant county gives a stable mapping.

rk_county = (ppr.dropna(subset=["routing_key"])
             .groupby("routing_key")["county"]
             .agg(lambda s: s.mode().iat[0]))

ppr["rk_county"] = ppr["routing_key"].map(rk_county)

# how often does the filed county disagree with the routing key's dominant county?
has_rk = ppr["routing_key"].notna()
mismatch = has_rk & (ppr["county"] != ppr["rk_county"])
print(f"County disagrees with routing key's dominant county: {mismatch.sum():,} ({mismatch.sum()/has_rk.sum()*100:.1f}%)")

County disagrees with routing key's dominant county: 10,950 (4.8%)


In [22]:
# how concentrated is each routing key in its dominant county?
purity = (ppr.dropna(subset=["routing_key"])
          .groupby("routing_key")["county"]
          .agg(lambda s: s.value_counts(normalize=True).iat[0]))

print(purity.describe().round(3).to_string())
print(f"\nLeast concentrated routing keys:")
print(purity.nsmallest(15).round(3).to_string())

count    302.000
mean       0.953
std        0.124
min        0.286
25%        0.992
50%        1.000
75%        1.000
max        1.000

Least concentrated routing keys:
routing_key
A65    0.286
A00    0.500
A12    0.500
E15    0.500
E23    0.500
E42    0.500
F49    0.500
K91    0.500
N11    0.500
P91    0.500
T67    0.500
V12    0.500
X00    0.500
X95    0.500
Y94    0.500


In [23]:
rk_counts = ppr["routing_key"].value_counts()

low_purity = purity.nsmallest(20).index
print(pd.DataFrame({
    "purity": purity[low_purity].round(3),
    "n_sales": rk_counts[low_purity]
}).to_string())

             purity  n_sales
routing_key                 
A65           0.286        7
A00           0.500        2
A12           0.500        2
E15           0.500        2
E23           0.500        2
E42           0.500        2
F49           0.500        2
K91           0.500        2
N11           0.500        2
P91           0.500        2
T67           0.500        2
V12           0.500        2
X00           0.500        2
X95           0.500        2
Y94           0.500        2
A82           0.596     2022
A42           0.600       75
E32           0.636      522
A92           0.661     4625
N93           0.667        3


In [24]:
MIN_SALES = 30
reliable = purity[rk_counts.reindex(purity.index) >= MIN_SALES]

print(f"Routing keys with >= {MIN_SALES} sales: {len(reliable)} of {len(purity)}")
print(f"\nPurity distribution:")
print(reliable.describe().round(3).to_string())
print(f"\nLeast concentrated:")
print(pd.DataFrame({
    "purity": reliable.nsmallest(12).round(3),
    "n_sales": rk_counts[reliable.nsmallest(12).index]
}).to_string())

Routing keys with >= 30 sales: 139 of 302

Purity distribution:
count    139.000
mean       0.958
std        0.079
min        0.596
25%        0.958
50%        0.995
75%        0.998
max        1.000

Least concentrated:
             purity  n_sales
routing_key                 
A82           0.596     2022
A42           0.600       75
E32           0.636      522
A92           0.661     4625
F52           0.751      635
N37           0.756     2121
F91           0.802     3058
P51           0.817     2684
W91           0.820     4350
N41           0.833     1713
F26           0.848     1706
A81           0.854      513


In [25]:
# what does the A92 split actually look like?
a92 = ppr[ppr["routing_key"] == "A92"]
print(a92["county"].value_counts().to_string())

county
Louth      3059
Meath      1558
Dublin        5
Mayo          1
Cavan         1
Donegal       1


In [26]:
# Cross-county routing keys

# routing keys are built around postal delivery areas and legitimately
# straddle county boundaries where towns sit near a border

border_keys = reliable[reliable < 0.9].sort_values()
border = pd.DataFrame({
    "purity": border_keys.round(3),
    "n_sales": rk_counts[border_keys.index],
    "dominant_county": rk_county[border_keys.index],
})
print("Routing keys spanning counties (>=30 sales, purity <0.9):")
print(border.to_string())

Routing keys spanning counties (>=30 sales, purity <0.9):
             purity  n_sales dominant_county
routing_key                                 
A82           0.596     2022           Cavan
A42           0.600       75          Dublin
E32           0.636      522       Tipperary
A92           0.661     4625           Louth
F52           0.751      635       Roscommon
N37           0.756     2121       Westmeath
F91           0.802     3058           Sligo
P51           0.817     2684            Cork
W91           0.820     4350         Kildare
N41           0.833     1713         Leitrim
F26           0.848     1706            Mayo
A81           0.854      513        Monaghan
F45           0.855     1188       Roscommon
V94           0.855     8428        Limerick
P36           0.862      803            Cork
K32           0.863     1550          Dublin
F35           0.877      399            Mayo
R93           0.884     2498          Carlow
P56           0.886      438            Co

In [27]:
# How many county labels are likely typos? 

# within each routing key, counties accounting for a trivial share
# of sales are likely data entry errors rather than real geography

rk_county_counts = (ppr.dropna(subset=["routing_key"])
                    .groupby(["routing_key", "county"])
                    .size()
                    .rename("n")
                    .reset_index())

rk_totals = rk_county_counts.groupby("routing_key")["n"].transform("sum")
rk_county_counts["share"] = rk_county_counts["n"] / rk_totals

# a county with under 1% of a routing key's sales, and fewer than 10 sales,
# is very unlikely to be genuine geography
suspect = rk_county_counts[(rk_county_counts["share"] < 0.01) &
                           (rk_county_counts["n"] < 10)]

print(f"Suspect routing key / county pairs: {len(suspect):,}")
print(f"Sales affected: {suspect['n'].sum():,} "
      f"({suspect['n'].sum() / ppr['routing_key'].notna().sum() * 100:.2f}% of Eircoded sales)")

Suspect routing key / county pairs: 412
Sales affected: 611 (0.27% of Eircoded sales)


In [28]:
# flag rather than delete or overwrite
suspect_pairs = set(zip(suspect["routing_key"], suspect["county"]))
ppr["county_suspect"] = [
    (rk, c) in suspect_pairs
    for rk, c in zip(ppr["routing_key"], ppr["county"])
]
print(f"Flagged: {ppr['county_suspect'].sum():,}")

Flagged: 611


## 7. Save Cleaned PPR

In [32]:
keep_cols = [
    "date", "year", "month", "address", "county",
    "eircode", "routing_key", "rk_county",
    "price", "price_incl_vat", "property_type",
    "not_full_market", "vat_exclusive", "county_suspect"
]

ppr_clean = ppr[keep_cols].copy()
ppr_clean.to_csv(PROCESSED / "ppr_clean.csv", index=False)

print(f"Saved {len(ppr_clean):,} rows to ppr_clean.csv")
print(f"\nColumns: {list(ppr_clean.columns)}")
print(f"\nMemory: {ppr_clean.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")

Saved 750,660 rows to ppr_clean.csv

Columns: ['date', 'year', 'month', 'address', 'county', 'eircode', 'routing_key', 'rk_county', 'price', 'price_incl_vat', 'property_type', 'not_full_market', 'vat_exclusive', 'county_suspect']

Memory: 331.3 MB


In [33]:
size_mb = (PROCESSED / "ppr_clean.csv").stat().st_size / 1_048_576
print(f"ppr_clean.csv on disk: {size_mb:.1f} MB")

ppr_clean.csv on disk: 86.2 MB


In [34]:
# option 2 - parquet, smaller still and preserves dtypes
ppr_clean.to_parquet(PROCESSED / "ppr_clean.parquet", index=False)
print(f"parquet: {(PROCESSED / 'ppr_clean.parquet').stat().st_size / 1_048_576:.1f} MB")

parquet: 23.2 MB
